## Download a single DTM + DSM tile

Independent of the merge above: just one BEV tile, for both products, covering a
point of interest. `fetch_bev_als.point_to_tile_id()` maps the point to its
containing tile (doesn't handle points near a tile border - just picks the single
containing tile, which is fine here).

In [ ]:
import sys

# This notebook lives in scripts/; the repo root holds the shared util/ package.
sys.path.insert(0, "..")

from fetch_bev_als import download_tiles, point_to_tile_id

# Großglockner
GG_LAT = 47.0742
GG_LON = 12.6938
# Stephansdom
#GG_LAT = 48.208492
#GG_LON = 16.373127
ZOOM = 17

GG_TILE_ID = point_to_tile_id(GG_LAT, GG_LON)
DTM_DIR = "../data/als_tiles/DTM"
DSM_DIR = "../data/als_tiles/DSM"

dtm_tile_path = download_tiles("DTM", [GG_TILE_ID], DTM_DIR)[0]
dsm_tile_path = download_tiles("DSM", [GG_TILE_ID], DSM_DIR)[0]
dtm_tile_path, dsm_tile_path

## Evaluate Tile Bounds

In [ ]:
import math

import rasterio.warp


def deg2tile(lat: float, lon: float, z: int) -> tuple[int, int]:
    """WGS84 lon/lat -> XYZ (Google/OSM scheme) tile indices at zoom z."""
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    lat_rad = math.radians(lat)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1.0 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def tile_bounds_lonlat(x: int, y: int, z: int) -> tuple[float, float, float, float]:
    """XYZ tile indices -> (west, south, east, north) in WGS84 degrees."""
    n = 2 ** z

    def lat_of_row(row):
        return math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * row / n))))

    west = x / n * 360.0 - 180.0
    east = (x + 1) / n * 360.0 - 180.0
    north = lat_of_row(y)
    south = lat_of_row(y + 1)
    return west, south, east, north


tile_x, tile_y = deg2tile(GG_LAT, GG_LON, ZOOM)
bounds_lonlat = tile_bounds_lonlat(tile_x, tile_y, ZOOM)
bounds_3857 = rasterio.warp.transform_bounds("EPSG:4326", "EPSG:3857", *bounds_lonlat)

print(f"tile z={ZOOM} x={tile_x} y={tile_y}")
print(f"bounds (lon/lat): {bounds_lonlat}")
print(f"bounds (EPSG:3857): {bounds_3857}")

## Reproject into the 256x256 Web Mercator grid (+1px apron)

Two things here differ from a naive per-tile reprojection, both because this
notebook is a rehearsal for `tile_creators/als_normals.py`:

**A 1-pixel apron.** The gradient at a tile's edge pixel needs its neighbour
*outside* the tile. Clamping at the border (`np.pad(..., mode="edge")`) fabricates
a zero-gradient neighbour and leaves a visible seam along every tile boundary. So
the destination grid is 258x258 covering the tile bounds expanded by one pixel;
the normal functions consume the apron and return exactly 256x256.

**Real nodata.** BEV coverage stops at the Austrian border and the source has
voids, so a zero-initialized destination silently turns "no data" into "sea level"
and manufactures a cliff. The destination is initialized to NaN instead, and the
validity mask is carried alongside the heights.

In [ ]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.warp import reproject

TILE_SIZE = 256
APRON = 1  # one extra pixel per side, so the edge gradient has a real neighbour

# Expand the tile bounds by exactly one destination pixel on every side.
_px = (bounds_3857[2] - bounds_3857[0]) / TILE_SIZE
apron_bounds_3857 = (
    bounds_3857[0] - _px * APRON,
    bounds_3857[1] - _px * APRON,
    bounds_3857[2] + _px * APRON,
    bounds_3857[3] + _px * APRON,
)
APRON_SIZE = TILE_SIZE + 2 * APRON
dst_transform = from_bounds(*apron_bounds_3857, APRON_SIZE, APRON_SIZE)


def reproject_to_tile(src_path: str) -> np.ndarray:
    """-> (APRON_SIZE, APRON_SIZE) float32 heights in meters, NaN where there is
    no source data. NaN rather than 0 matters: outside BEV coverage a zero would
    read as sea level and produce a fake cliff at every border tile."""
    with rasterio.open(src_path) as src:
        # init_dest_nodata defaults to True, so anything the warp doesn't touch
        # (i.e. outside the source footprint) keeps dst_nodata.
        dst = np.full((APRON_SIZE, APRON_SIZE), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=dst_transform,
            dst_crs="EPSG:3857",
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
        return dst


def stencil_valid(height_apron: np.ndarray) -> np.ndarray:
    """(APRON_SIZE, APRON_SIZE) heights -> (TILE_SIZE, TILE_SIZE) bool. A normal
    is only trusted where its full 3x3 height stencil was valid, so a void never
    leaks a fabricated gradient into a neighbouring real pixel."""
    ok = np.isfinite(height_apron)
    h, w = ok.shape
    out = np.ones((h - 2, w - 2), dtype=bool)
    for dy in range(3):
        for dx in range(3):
            out &= ok[dy:dy + h - 2, dx:dx + w - 2]
    return out


dtm_arr = reproject_to_tile(dtm_tile_path)
dsm_arr = reproject_to_tile(dsm_tile_path)
dtm_valid = stencil_valid(dtm_arr)
dsm_valid = stencil_valid(dsm_arr)

# If a source uses an undeclared sentinel (-9999 etc.) instead of a real nodata
# tag, the valid fraction will read 100% and the height range will look absurd.
print(f"DTM {dtm_arr.shape} valid {dtm_valid.mean():6.1%}  "
      f"range {np.nanmin(dtm_arr):.1f}..{np.nanmax(dtm_arr):.1f} m")
print(f"DSM {dsm_arr.shape} valid {dsm_valid.mean():6.1%}  "
      f"range {np.nanmin(dsm_arr):.1f}..{np.nanmax(dsm_arr):.1f} m")

## Web Mercator ground-scale correction (altitude vs. quad size)

Web Mercator inflates horizontal distances away from the equator by `1/cos(lat)`.
weBIGeo's renderer corrects for this by scaling the *height* sample instead of the
horizontal quad size:

```wgsl
let world_space_y: f32 = (*position).y + camera.position.y;
let altitude_correction_factor: f32 = 0.125 / cos(y_to_lat(world_space_y));  // AlpineMapsOrg/renderer#5
let adjusted_altitude: f32 = altitude_tex * altitude_correction_factor;
```

Mathematically this is equivalent to what this notebook did before (scaling
`quad_width`/`quad_height` by `cos(lat)`) — either way the height/distance ratio
that drives the normal direction ends up the same. To match weBIGeo exactly,
`quad_width`/`quad_height` below are now the *raw*, uncorrected Web Mercator
meters/pixel, and the correction is applied to the height values instead via
`altitude_correction_factor = 1 / cos(lat)` (their `0.125` is a raw-texture-value
-to-meters unit scale that doesn't apply here since `dtm_arr`/`dsm_arr` are
already float meters).

`APPLY_ALTITUDE_CORRECTION` toggles it on/off — flip it and re-run the cells below
to compare `dtm_h`/`dsm_h` (used everywhere downstream instead of the raw
`dtm_arr`/`dsm_arr`) with and without the correction.

In [ ]:
APPLY_ALTITUDE_CORRECTION = True  # flip and re-run downstream cells to compare

WEB_MERCATOR_RES_ZOOM0 = 156543.03392804097  # meters/pixel at zoom 0, equator
quad_width = quad_height = WEB_MERCATOR_RES_ZOOM0 / 2 ** ZOOM  # raw, uncorrected

# weBIGeo folds the Web Mercator latitude distortion into the height value
# instead of the horizontal quad size - see AlpineMapsOrg/renderer#5. Their 0.125
# factor also converts a raw quantized height-texture value to meters, which
# doesn't apply here since dtm_arr/dsm_arr are already float meters.
altitude_correction_factor = 1.0 / math.cos(math.radians(GG_LAT))


def corrected_height(height: np.ndarray) -> np.ndarray:
    return height * altitude_correction_factor if APPLY_ALTITUDE_CORRECTION else height


dtm_h = corrected_height(dtm_arr)
dsm_h = corrected_height(dsm_arr)

print(f"quad_width = quad_height = {quad_width:.4f} m (raw Web Mercator, uncorrected)")
print(f"altitude_correction_factor = {altitude_correction_factor:.4f} (applied={APPLY_ALTITUDE_CORRECTION})")

## Fetch the matching basemap.at orthophoto tile

Same `z/x/y` tile, real orthophoto imagery from basemap.at (same source/URL scheme as `tile_creators/debug_ortho.py`). A visual reference to sanity-check the normal maps against actual terrain features (ridges, rock, snow/ice) visible in the photo.

In [ ]:
import io
import os

import requests
from PIL import Image

BASEMAP_URL_TEMPLATE = "https://gataki.cg.tuwien.ac.at/raw/basemap/tiles/{z}/{y}/{x}.jpeg"
ORTHO_DIR = "../cache/tmp_normal_test/ortho"


def fetch_basemap_tile(z: int, x: int, y: int) -> Image.Image:
    """Download the tile (skipping if already cached on disk) and return it as an image."""
    os.makedirs(ORTHO_DIR, exist_ok=True)
    path = os.path.join(ORTHO_DIR, f"{z}_{x}_{y}.jpeg")
    if os.path.exists(path):
        return Image.open(path).convert("RGB")
    url = BASEMAP_URL_TEMPLATE.format(z=z, y=y, x=x)
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    return Image.open(io.BytesIO(r.content)).convert("RGB")


ortho_img = fetch_basemap_tile(ZOOM, tile_x, tile_y)
ortho_img

## Normal map: weBIGeo finite-difference method

Direct port of `normal_by_finite_difference_method` from weBIGeo's `normal.wgsl`
shader (the exact method the renderer uses at draw time). The shader's own
`altitude_correction_factor` (see the cell above) is applied upstream to produce
`dtm_h`/`dsm_h`, so it's implicitly `1.0` by the time it reaches this function.

This is a plain 4-neighbor (axis-aligned L/R/U/D only) central difference, *not*
the classic 3x3 Sobel kernel below — cheaper, no diagonal-neighbor blending.

Instead of padding, the function consumes the 258x258 apron array from above and
returns exactly 256x256, so every output pixel — including the border ones — is
computed from real neighbouring data.

### On the "asymmetric" sign convention

An earlier version of this notebook called the shader's pairing (`nx = hL - hR`,
but `ny = hD - hU` with no swap) a quirk of the stackoverflow-derived formula.
It isn't a quirk — it's correct, and the asymmetry is only apparent. Slippy-map
rows increase *southward*, so `padded[2:]` (`hD`) is the **south** neighbour and
`padded[:-2]` (`hU`) is the **north** one. That makes

    nx = h_west  - h_east  = -dh/dx_east  * 2*quad_width
    ny = h_south - h_north = -dh/dy_north * 2*quad_height

and with `nz = 2`, the vector `(nx, ny, 2)` is proportional to
`(-dh/dx, -dh/dy, 1)` — a consistent ENU normal (+X east, +Y north, +Z up). The
`2` is not arbitrary either: the differences span two cells but are divided by
one `quad_width`, so it cancels exactly. Both this and the Sobel variant below
are correctly scaled and directly comparable.

In [ ]:
def normal_by_finite_difference(height_apron: np.ndarray, quad_width: float, quad_height: float) -> np.ndarray:
    """(APRON_SIZE, APRON_SIZE) heights -> (TILE_SIZE, TILE_SIZE, 3) ENU normals."""
    h = np.where(np.isfinite(height_apron), height_apron, 0.0)
    hL = h[1:-1, :-2]
    hR = h[1:-1, 2:]
    hD = h[2:, 1:-1]
    hU = h[:-2, 1:-1]

    nx = (hL - hR) / quad_width
    ny = (hD - hU) / quad_height
    nz = np.full_like(nx, 2.0)

    normal = np.stack([nx, ny, nz], axis=-1)
    normal /= np.linalg.norm(normal, axis=-1, keepdims=True)
    return normal


def normal_to_rgb(normal: np.ndarray) -> np.ndarray:
    """Display-only xyz->rgb mapping. NOT the storage format - see the encoding
    section at the end of the notebook for that."""
    return ((normal * 0.5 + 0.5) * 255).clip(0, 255).astype(np.uint8)


dtm_normal_fd = normal_by_finite_difference(dtm_h, quad_width, quad_height)
dsm_normal_fd = normal_by_finite_difference(dsm_h, quad_width, quad_height)
dtm_normal_fd_rgb = normal_to_rgb(dtm_normal_fd)
dsm_normal_fd_rgb = normal_to_rgb(dsm_normal_fd)

## Normal map: standard Sobel operator (for comparison)

Classic 3x3 Sobel kernels (blends the 4 diagonal neighbors too, unlike the
finite-difference method above), same real-world `quad_width`/`quad_height` so the
two are on the same scale, and the same 258x258 apron input. No scipy dependency
needed (the `clouds` conda env doesn't have it) — plain numpy slicing.

Same ENU convention as above: `dzdy` here is the derivative along increasing rows,
i.e. *southward*, so `+dzdy` already equals `-dh/dy_north` and needs no extra
negation. `gx / 8` is the standard Sobel normalization and yields the height
difference per cell, so dividing by one `quad_width` gives true meters/meter —
directly comparable with the finite-difference result.

The interesting number is how far the two methods disagree: that spread is an
empirical proxy for how much noise the source data carries into the normal, and
it's what justifies spending only 8 bits per component on the encoding.

In [ ]:
def normal_by_sobel(height_apron: np.ndarray, quad_width: float, quad_height: float) -> np.ndarray:
    """(APRON_SIZE, APRON_SIZE) heights -> (TILE_SIZE, TILE_SIZE, 3) ENU normals."""
    h = np.where(np.isfinite(height_apron), height_apron, 0.0)
    tl, tm, tr = h[:-2, :-2], h[:-2, 1:-1], h[:-2, 2:]
    ml, mr = h[1:-1, :-2], h[1:-1, 2:]
    bl, bm, br = h[2:, :-2], h[2:, 1:-1], h[2:, 2:]

    gx = -(tl + 2 * ml + bl) + (tr + 2 * mr + br)
    gy = -(tl + 2 * tm + tr) + (bl + 2 * bm + br)

    dzdx = (gx / 8.0) / quad_width
    dzdy = (gy / 8.0) / quad_height

    # dzdy is the southward derivative (rows increase south), so it already is
    # -dh/dy_north and is used un-negated - same ENU frame as the method above.
    normal = np.stack([-dzdx, dzdy, np.ones_like(dzdx)], axis=-1)
    normal /= np.linalg.norm(normal, axis=-1, keepdims=True)
    return normal


dtm_normal_sobel = normal_by_sobel(dtm_h, quad_width, quad_height)
dsm_normal_sobel = normal_by_sobel(dsm_h, quad_width, quad_height)
dtm_normal_sobel_rgb = normal_to_rgb(dtm_normal_sobel)
dsm_normal_sobel_rgb = normal_to_rgb(dsm_normal_sobel)

## Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
rows = [
    ("DTM", dtm_h[APRON:-APRON, APRON:-APRON], dtm_normal_fd_rgb, dtm_normal_sobel_rgb),
    ("DSM", dsm_h[APRON:-APRON, APRON:-APRON], dsm_normal_fd_rgb, dsm_normal_sobel_rgb),
]
for row, (label, height, fd_rgb, sobel_rgb) in enumerate(rows):
    axes[row, 0].imshow(ortho_img)
    axes[row, 0].set_title("basemap.at orthophoto")
    axes[row, 1].imshow(height, cmap="gray")
    axes[row, 1].set_title(f"{label} height")
    axes[row, 2].imshow(fd_rgb)
    axes[row, 2].set_title(f"{label} normal (weBIGeo finite-diff)")
    axes[row, 3].imshow(sobel_rgb)
    axes[row, 3].set_title(f"{label} normal (Sobel)")
    for ax in axes[row]:
        ax.axis("off")
fig.suptitle(f"z={ZOOM} x={tile_x} y={tile_y} — {GG_LAT:.4f}, {GG_LON:.4f}")
plt.tight_layout()
plt.show()

## 3D check: drape the orthophoto over the DSM (interactive)

Extrudes the DSM heightfield and textures it with the basemap.at orthophoto,
rotatable/zoomable via Plotly (`pip install plotly` if not already in this
kernel's env). Since both arrays are already resampled onto the exact same
256x256 grid, any misalignment between the DTM/DSM reprojection and the ortho
fetch (wrong bounds, swapped axes, off-by-one tile index, etc.) would show up
here as image features (ridgelines, buildings, rock/snow boundaries) not lining
up with the 3D bumps.

Plotly's `Surface` only maps a single scalar field through one colorscale (no
true RGB texture support like matplotlib's `facecolors`), so the ortho image is
quantized to a 256-color palette (`Image.quantize`) and that palette becomes a
custom colorscale — a lossy approximation, but close enough to still recognize
features by color/shape. `X`/`Y` use the same latitude-corrected
`quad_width`/`quad_height` as the normal maps above, so the plot is close to
true-to-scale (`Z_EXAGGERATION` is purely cosmetic and doesn't affect alignment).

In [ ]:
import plotly.graph_objects as go

Z_EXAGGERATION = 1.0  # cosmetic only - doesn't affect X/Y alignment

rows = np.arange(TILE_SIZE)
cols = np.arange(TILE_SIZE)
xs = cols * quad_width
ys = (TILE_SIZE - 1 - rows) * quad_height  # flip so north is "up"
Z = dsm_h[APRON:-APRON, APRON:-APRON] * Z_EXAGGERATION  # drop the apron: xs/ys are TILE_SIZE

ortho_quantized = ortho_img.resize((TILE_SIZE, TILE_SIZE)).quantize(colors=256)
palette = ortho_quantized.getpalette()[:256 * 3]
colorscale = [
    [i / 255, f"rgb({palette[i * 3]},{palette[i * 3 + 1]},{palette[i * 3 + 2]})"]
    for i in range(256)
]
surfacecolor = np.array(ortho_quantized, dtype=np.float64)

fig = go.Figure(data=[go.Surface(
    x=xs, y=ys, z=Z,
    surfacecolor=surfacecolor,
    colorscale=colorscale,
    cmin=0, cmax=255,
    showscale=False,
)])
fig.update_layout(
    title="DSM extruded, textured with the basemap.at orthophoto (256-color quantized)",
    scene=dict(aspectmode="manual", aspectratio=dict(x=1, y=1, z=1)),
    width=800, height=800,
)
fig.show()

# Encoding: how to store these normals in a PNG

Everything above produces float normals. This section measures the format the
tile source actually writes, implemented in `util/encoding.py`:

**hemi-octahedral projection -> 127-centred 8-bit quantization -> R/G of an RGB
PNG** (B reserved, no alpha channel).

The three questions worth answering with numbers rather than intuition:

1. How much angular error does 8 bits per component cost, and how does that
   compare to 16 bits (the original proposal) and to plain octahedral?
2. How large is that error next to the *noise already in the data*? If the source
   normals are uncertain by degrees, precision below that is measuring nothing.
3. What does the extra precision cost in bytes?

## 1. Round-trip angular error

`hemi-oct 8:8` is the tile format. `plain oct 8:8` is the same bit budget using
weBIGeo's existing `v3f32_to_oct`, for reference. `plain oct 16:16` is the
original RGBA-packed proposal.

Heightfield normals always point up (`n.z >= 0`), so plain octahedral confines
every value to the `|x|+|y| <= 1` diamond and leaves the square's corners unused.
The hemi-oct 45° rotation fills the whole square instead — the expected gain is
sqrt(2), and it also removes the octahedral fold from the sampled domain, so GPU
bilinear filtering between two encoded texels can never interpolate across it.

In [ ]:
from util import encoding


def report_roundtrip(label: str, normal: np.ndarray, valid: np.ndarray) -> None:
    reference = normal[valid]
    variants = []

    # The tile format.
    decoded = encoding.decode_normals(encoding.encode_normals(normal))
    variants.append(("hemi-oct 8:8  (tile format)", decoded))

    # Plain oct, conventional unorm mapping, at both bit depths.
    e = encoding.normal_to_oct(normal)
    for bits, scale in ((8, 255), (16, 65535)):
        q = np.rint((e * 0.5 + 0.5) * scale)
        variants.append((f"plain oct {bits}:{bits}", encoding.oct_to_normal(q / scale * 2.0 - 1.0)))

    print(f"--- {label}  ({valid.sum()} valid px) ---")
    for name, decoded in variants:
        err = encoding.angle_between_deg(decoded[valid], reference)
        print(f"  {name:28s} mean {err.mean():7.4f}   p99 {np.percentile(err, 99):7.4f}   max {err.max():7.4f}   deg")


report_roundtrip("DTM / finite difference", dtm_normal_fd, dtm_valid)
report_roundtrip("DSM / finite difference", dsm_normal_fd, dsm_valid)
report_roundtrip("DTM / Sobel", dtm_normal_sobel, dtm_valid)
report_roundtrip("DSM / Sobel", dsm_normal_sobel, dsm_valid)

## 2. The flat normal, and why the quantization is centred on 127

The conventional unorm mapping `round((e * 0.5 + 0.5) * 255)` spreads 256 codes
over 255 intervals, so the midpoint lands *between* two codes: a perfectly flat
surface encodes to 128 and decodes to `e = 0.0039`, i.e. tilted by 0.225°.

That is far below the data's own noise — but unlike noise it's a *systematic*
bias in one fixed direction, so it never averages out. A whole lake, glacier
plateau or flat roof gets the same false shading offset across its entire
surface, and the mipmap pyramid preserves it at every zoom level.

Centring on 127 with a half-range of 127 puts `e = 0` exactly on a code, and
makes the encoding symmetric under negation (mirrored slopes land equidistant
from 127). It costs one unused code (255) and a 0.4% coarser step.

In [ ]:
flat = np.array([[[0.0, 0.0, 1.0]]])

rgb_centred = encoding.encode_normals(flat)
err_centred = encoding.angle_between_deg(encoding.decode_normals(rgb_centred), flat)[0, 0]

e = encoding.normal_to_hemioct(flat)
rgb_conventional = np.rint((e * 0.5 + 0.5) * 255)
err_conventional = encoding.angle_between_deg(
    encoding.hemioct_to_normal(rgb_conventional / 255.0 * 2.0 - 1.0), flat)[0, 0]

print(f"127-centred (used)  -> R,G = {rgb_centred[0, 0, :2].tolist()}   error {err_centred:.4f} deg")
print(f"conventional unorm  -> R,G = {rgb_conventional[0, 0].astype(int).tolist()}   error {err_conventional:.4f} deg")

assert err_centred == 0.0, "the flat normal must round-trip exactly"

## 3. The noise floor: how far do the two gradient methods disagree?

This is the number that decides the bit depth. Finite-difference and Sobel are
both defensible estimators of the same underlying surface, computed from the same
heights on the same grid — so where they disagree, the disagreement comes from
the data, not from the encoding.

If that spread is on the order of degrees, then a quantization error of a few
tenths of a degree is comfortably below the noise floor and 16 bits per component
would be measuring nothing that is actually known about the terrain.

In [ ]:
for label, fd, sobel, valid in [
    ("DTM", dtm_normal_fd, dtm_normal_sobel, dtm_valid),
    ("DSM", dsm_normal_fd, dsm_normal_sobel, dsm_valid),
]:
    spread = encoding.angle_between_deg(fd[valid], sobel[valid])
    print(f"{label}: finite-diff vs Sobel   mean {spread.mean():7.3f}   "
          f"p99 {np.percentile(spread, 99):7.3f}   max {spread.max():7.3f}   deg")

## 4. What the extra precision costs in bytes

The low byte of a value quantized from noisy LiDAR is close to random, so PNG
stores it nearly raw. The DSM tile is the interesting one: buildings and
vegetation give it far more high-frequency content than the DTM, which is where
an incompressible low byte hurts most.

Multiply the per-tile figure by the tile count for a zoom level to get the real
storage cost of a full tileset (~59k tiles at z17 per BEV source file, ~15k at
z16) — that's the number that decides `MAX_ZOOM`.

In [ ]:
import io as _io


def png_size(arr: np.ndarray, mode: str) -> int:
    buf = _io.BytesIO()
    Image.fromarray(arr, mode=mode).save(buf, format="PNG")
    return len(buf.getvalue())


def report_size(label: str, normal: np.ndarray, valid: np.ndarray) -> None:
    hemioct8 = encoding.encode_normals(normal, valid)

    # The original proposal: oct 16:16 packed as R/G = hi/lo of x, B/A = hi/lo of y.
    q = np.rint((encoding.normal_to_oct(normal) * 0.5 + 0.5) * 65535).astype(np.uint16)
    oct16 = np.dstack([
        (q[..., 0] >> 8).astype(np.uint8), (q[..., 0] & 0xFF).astype(np.uint8),
        (q[..., 1] >> 8).astype(np.uint8), (q[..., 1] & 0xFF).astype(np.uint8),
    ])

    naive = normal_to_rgb(normal)  # xyz straight into rgb, 3 channels, no projection

    base = png_size(hemioct8, "RGB")
    print(f"--- {label} ---")
    for name, arr, mode in [
        ("hemi-oct 8:8   RGB  (used)", hemioct8, "RGB"),
        ("plain oct 16:16 RGBA", oct16, "RGBA"),
        ("naive xyz->rgb  RGB", naive, "RGB"),
    ]:
        size = png_size(arr, mode)
        print(f"  {name:28s} {size:7d} B   ({size / base:4.2f}x)")


report_size("DTM / Sobel", dtm_normal_sobel, dtm_valid)
report_size("DSM / Sobel", dsm_normal_sobel, dsm_valid)

## 5. Visual check

Four panels per product: the float normals, **the encoded tile exactly as it is
stored in the PNG**, those bytes decoded back to normals, and the per-pixel
angular error between original and decoded.

The encoded column is worth looking at directly, because it does *not* look like
a conventional normal map. There is no blue channel (B is reserved and zero), and
the hemi-octahedral 45° rotation means R and G are not x and y — a slope facing
north-east moves both channels together rather than one alone. Flat ground sits at
(127, 127), so large flat areas read as a uniform olive tone, and the nodata fill
is that exact same value.

Quantization error should look like fine uncorrelated speckle. Structure in the
error map — banding, or error concentrated on particular slope orientations —
would mean the projection is distorting somewhere rather than the quantization
simply rounding.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for row, (label, normal, valid) in enumerate([
    ("DTM", dtm_normal_sobel, dtm_valid),
    ("DSM", dsm_normal_sobel, dsm_valid),
]):
    encoded = encoding.encode_normals(normal, valid)   # what lands in the PNG
    decoded = encoding.decode_normals(encoded)
    err = encoding.angle_between_deg(decoded, normal)

    axes[row, 0].imshow(normal_to_rgb(normal))
    axes[row, 0].set_title(f"{label} normal (float)")
    axes[row, 1].imshow(encoded)
    axes[row, 1].set_title(f"{label} encoded tile (R=hemioct.x, G=hemioct.y, B=0)")
    axes[row, 2].imshow(normal_to_rgb(decoded))
    axes[row, 2].set_title(f"{label} normal (decoded from tile)")
    im = axes[row, 3].imshow(np.where(valid, err, np.nan), cmap="magma")
    axes[row, 3].set_title(f"{label} angular error [deg]")
    fig.colorbar(im, ax=axes[row, 3], fraction=0.046)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

# 6. WebP instead of PNG, and cheaper PNGs

The tile format is PNG because the renderer cannot decode WebP. This section
measures what that constraint costs, because the z17 storage projection makes it
worth knowing: at ~94 KB/tile a single 50x50 km BEV source file is ~7.4 GB
(z17 + pyramid), and Austria-wide is a few hundred GB per product.

**Lossless WebP** puts a number on the constraint. `docs/webp-instead-png.md`
measured ~45% for the `cosmos_snow` tiles, but that format is different (RGBA, a
smooth quantized scalar field) so it doesn't transfer — normal tiles are two
high-entropy channels.

**PNG variants** are the only lever actually available today. `optimize` and
`compress_level` cost encode time only. The `LA` variant is more interesting:
our format has just two real channels, and PNG's grayscale+alpha colour type
stores exactly two, dropping the reserved B channel from the file entirely.
Caveat if that looks tempting — `LA` puts the second component in the *alpha*
channel, which reintroduces the browser premultiplication hazard that choosing a
3-channel RGB tile was meant to avoid.

Every variant is round-trip checked with `np.array_equal`. These channels are
vector components, not pixels, so a format that doesn't reproduce the bytes
exactly is not a candidate regardless of size.

In [ ]:
def encode_size(arr: np.ndarray, mode: str, fmt: str, **kwargs) -> tuple[int, bool]:
    """-> (bytes on disk, round-tripped exactly). Reads the encoded blob back and
    compares byte-for-byte: these channels are vector components, so 'looks the
    same' is not good enough."""
    buf = _io.BytesIO()
    Image.fromarray(arr, mode=mode).save(buf, format=fmt, **kwargs)
    blob = buf.getvalue()
    back = np.array(Image.open(_io.BytesIO(blob)))
    return len(blob), np.array_equal(back, arr)


def report_lossless(label: str, normal: np.ndarray, valid: np.ndarray) -> None:
    rgb = encoding.encode_normals(normal, valid)
    la = np.ascontiguousarray(rgb[..., :2])  # drop the reserved B channel entirely

    variants = [
        ("PNG                  RGB", rgb, "RGB", "PNG", {}),
        ("PNG optimize+lvl9    RGB", rgb, "RGB", "PNG", dict(optimize=True, compress_level=9)),
        ("PNG 2-channel        LA ", la, "LA", "PNG", dict(optimize=True, compress_level=9)),
        # exact=True is what stops libwebp zeroing RGB behind transparent pixels.
        # Moot for a tile with no alpha, but kept so the call stays correct if the
        # format ever grows one.
        ("WebP lossless        RGB", rgb, "RGB", "WEBP", dict(lossless=True, exact=True)),
        ("WebP lossless m6     RGB", rgb, "RGB", "WEBP", dict(lossless=True, exact=True, quality=100, method=6)),
    ]

    base = None
    print(f"--- {label} ---")
    for name, arr, mode, fmt, kwargs in variants:
        size, exact = encode_size(arr, mode, fmt, **kwargs)
        base = base if base is not None else size
        flag = "exact" if exact else "LOSSY!"
        print(f"  {name:26s} {size:7d} B   ({size / base:4.2f}x)   {flag}")


report_lossless("DTM / Sobel", dtm_normal_sobel, dtm_valid)
report_lossless("DSM / Sobel", dsm_normal_sobel, dsm_valid)

# 7. BC5 (GPU block compression) instead of a PNG container

`docs/normal_map_encoding.md` flags BC5 as the option worth a number rather than
a guess: a GPU texture-compression format built for exactly this shape of data
(two independent 8-bit channels), decoded by sampling hardware instead of a
CPU-side PNG inflate.

No BC5 encoder is available in this env, so this simulates the algorithm
directly: each 4x4 block of a channel is re-quantized to 8 values interpolated
between that block's own min and max (BC4/BC5's `c0 > c1` mode - the one this
data always lands in, since an 8-bit hemi-oct byte hitting exact 0 or 255 is
rare). That mirrors what a real fast-path encoder does for smooth data; it does
not model the alternate 4-value+0/255 mode, so it's a slight overestimate of the
true error. The size, in contrast, is exact - BC5 is fixed-rate, 16 bytes per
4x4-texel block covering both channels, independent of content.

In [ ]:
def bc4_quantize(channel: np.ndarray, block: int = 4) -> np.ndarray:
    """(H, W) uint8 -> (H, W) uint8, simulating one BC4 channel: each block is
    re-quantized to 8 values interpolated between that block's own min and max
    (nearest-code selection), the same as a fast-path BC4/BC5 encoder."""
    h, w = channel.shape
    b = channel.reshape(h // block, block, w // block, block).transpose(0, 2, 1, 3)
    b = b.reshape(-1, block * block).astype(np.float64)
    c0 = b.max(axis=1, keepdims=True)
    c1 = b.min(axis=1, keepdims=True)
    codes = c1 + (c0 - c1) * (np.arange(8) / 7.0)  # (n_blocks, 8), degenerates to c1 on flat blocks
    idx = np.abs(b[:, :, None] - codes[:, None, :]).argmin(axis=-1)
    q = np.rint(np.take_along_axis(codes, idx, axis=1)).astype(np.uint8)
    return q.reshape(h // block, w // block, block, block).transpose(0, 2, 1, 3).reshape(h, w)


def report_bc5(label: str, normal: np.ndarray, valid: np.ndarray) -> None:
    hemioct8 = encoding.encode_normals(normal, valid)
    bc5_rg = np.stack([bc4_quantize(hemioct8[..., 0]), bc4_quantize(hemioct8[..., 1])], axis=-1)
    decoded = encoding.hemioct_to_normal(encoding.bytes_to_hemioct(bc5_rg))
    baseline = encoding.decode_normals(hemioct8)

    h, w = hemioct8.shape[:2]
    bc5_bytes = (h // 4) * (w // 4) * 16  # fixed-rate: 1 byte/texel, both channels
    png_bytes = png_size(hemioct8, "RGB")

    err_bc5 = encoding.angle_between_deg(decoded[valid], normal[valid])
    err_png = encoding.angle_between_deg(baseline[valid], normal[valid])

    print(f"--- {label} ---")
    print(f"  hemi-oct 8:8 PNG (current)   {png_bytes:7d} B            "
          f"mean {err_png.mean():6.4f}   p99 {np.percentile(err_png, 99):6.4f}   max {err_png.max():6.4f}  deg")
    print(f"  hemi-oct 8:8 BC5 (simulated) {bc5_bytes:7d} B  ({bc5_bytes / png_bytes:4.2f}x)   "
          f"mean {err_bc5.mean():6.4f}   p99 {np.percentile(err_bc5, 99):6.4f}   max {err_bc5.max():6.4f}  deg")


report_bc5("DTM / Sobel", dtm_normal_sobel, dtm_valid)
report_bc5("DSM / Sobel", dsm_normal_sobel, dsm_valid)